In [115]:
import pandas
from Dependency.File_converter import selecting
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import GridSearchCV,HalvingGridSearchCV

data_training, data_testing = selecting()

pandas.set_option('display.max_colwidth', None)

Forest = True
HistGB = False
SVM = False
#NaiveBayes always true to build default result

cols = ["params", "mean_test_score", "std_test_score", "rank_test_score"]

In [116]:
#Categorical_Naive_Bayes Grid

from sklearn.naive_bayes import CategoricalNB

p_grid = {
    "alpha":[0.5,1.0,2.0,5.0,10.0,15.0,20.0,25.0,30.0,40.0],
}

C_NB_grid = GridSearchCV(
    estimator=CategoricalNB(),
    param_grid=p_grid,
    scoring='accuracy',
    n_jobs=2)

def coord_positive(dataset):
    longitude = dataset["Longitude"]
    dataset["Longitude"] = longitude-dataset["Longitude"].min()
    longitude = dataset["Latitude"]
    dataset["Latitude"] = longitude-dataset["Latitude"].min()
    return dataset

data_training_positive = coord_positive(data_training)

data_training_X = data_training_positive.loc[ : , data_training_positive.columns != "Accident_Severity"]
data_training_Y = data_training_positive["Accident_Severity"]

C_NB_grid.fit(data_training_X, data_training_Y)

results = pandas.DataFrame(C_NB_grid.cv_results_)
    
results[cols].sort_values("rank_test_score").head(10)

c:\Code\Python\Python314\Lib\site-packages\sklearn\model_selection\_search.py:1234: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


,params,mean_test_score,std_test_score,rank_test_score
0,{'alpha': 0.5},NaN,NaN,1
1,{'alpha': 1.0},NaN,NaN,1
2,{'alpha': 2.0},NaN,NaN,1
3,{'alpha': 5.0},NaN,NaN,1
4,{'alpha': 10.0},NaN,NaN,1
5,{'alpha': 15.0},NaN,NaN,1
6,{'alpha': 20.0},NaN,NaN,1
7,{'alpha': 25.0},NaN,NaN,1
8,{'alpha': 30.0},NaN,NaN,1
9,{'alpha': 40.0},NaN,NaN,1


In [ ]:
#Random_Forest Grid

from sklearn.ensemble import RandomForestClassifier

p_grid = {
    #"n_estimators": [80,100,120,140],
    "criterion": ["gini", "entropy", "log_loss"],
    #"min_samples_split": [2,20,100],
    #"min_samples_leaf": [1,5,20],
    #"min_weight_fraction_leaf": [0.0,0.0001,0.001,0.01],
    "max_features": ["sqrt"],
    #"min_impurity_decrease": [0.0,0.0001,0.001,0.01],
    #"ccp_alpha": [0.0,0.1,1.0],
}

#HalvingGridSearchCV / GridSearchCV
Forest_grid = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=p_grid,
    scoring='accuracy',
    #max_resources=500,
    #max_resources=2000,
    n_jobs=4)

data_training_X = data_training.loc[ : , data_training.columns != "Accident_Severity"]
data_training_Y = data_training["Accident_Severity"]

if(Forest):
    Forest_grid.fit(data_training_X,data_training_Y)
    results = pandas.DataFrame(Forest_grid.cv_results_)

# Compact view of param, 10 worst
results[cols].sort_values("rank_test_score").tail(10)

,params,mean_test_score,std_test_score,rank_test_score
2,"{'criterion': 'entropy', 'max_features': 'sqrt'}",0.502756,0.004189,1
4,"{'criterion': 'log_loss', 'max_features': 'sqrt'}",0.502637,0.004699,2
5,"{'criterion': 'log_loss', 'max_features': 'log2'}",0.500622,0.005562,3
0,"{'criterion': 'gini', 'max_features': 'sqrt'}",0.500385,0.002664,4
3,"{'criterion': 'entropy', 'max_features': 'log2'}",0.500296,0.003306,5
1,"{'criterion': 'gini', 'max_features': 'log2'}",0.495674,0.003855,6


In [118]:
# Compact view of param, 10 best
results[cols].sort_values("rank_test_score").head(10)

,params,mean_test_score,std_test_score,rank_test_score
2,"{'criterion': 'entropy', 'max_features': 'sqrt'}",0.502756,0.004189,1
4,"{'criterion': 'log_loss', 'max_features': 'sqrt'}",0.502637,0.004699,2
5,"{'criterion': 'log_loss', 'max_features': 'log2'}",0.500622,0.005562,3
0,"{'criterion': 'gini', 'max_features': 'sqrt'}",0.500385,0.002664,4
3,"{'criterion': 'entropy', 'max_features': 'log2'}",0.500296,0.003306,5
1,"{'criterion': 'gini', 'max_features': 'log2'}",0.495674,0.003855,6


In [119]:
#Hist_Grabient_Boosting Grid

from sklearn.ensemble import HistGradientBoostingClassifier

p_grid = {
    "learning_rate":[0.03,0.04],
    "max_iter":[180],
    "max_leaf_nodes":[31],
    "min_samples_leaf":[30],
    "max_features":[0.4,0.5,0.6],
    "categorical_features":[["Day_of_Week","1st_Road_Class","Road_Type","Junction_Control","2nd_Road_Class","Pedestrian_Crossing-Physical_Facilities","Light_Conditions","Weather_Conditions","Road_Surface_Conditions","Urban_or_Rural_Area","Did_Police_Officer_Attend_Scene_of_Accident"]],
    "scoring":['accuracy'],
    "early_stopping":[False],
    "validation_fraction":[0.001],
}

#HalvingGridSearchCV / GridSearchCV
H_GB_grid = GridSearchCV(
    estimator=HistGradientBoostingClassifier(),
    param_grid=p_grid,
    scoring='accuracy',
    #max_resources=500,
    #max_resources=2000,
    n_jobs=2)

data_training_X = data_training.loc[ : , data_training.columns != "Accident_Severity"]
data_training_Y = data_training["Accident_Severity"]

if(HistGB):
    H_GB_grid.fit(data_training_X,data_training_Y)
    results = pandas.DataFrame(H_GB_grid.cv_results_)

# Compact view of param, 10 worst
results[cols].sort_values("rank_test_score").tail(10)


,params,mean_test_score,std_test_score,rank_test_score
2,"{'criterion': 'entropy', 'max_features': 'sqrt'}",0.502756,0.004189,1
4,"{'criterion': 'log_loss', 'max_features': 'sqrt'}",0.502637,0.004699,2
5,"{'criterion': 'log_loss', 'max_features': 'log2'}",0.500622,0.005562,3
0,"{'criterion': 'gini', 'max_features': 'sqrt'}",0.500385,0.002664,4
3,"{'criterion': 'entropy', 'max_features': 'log2'}",0.500296,0.003306,5
1,"{'criterion': 'gini', 'max_features': 'log2'}",0.495674,0.003855,6


In [120]:
# Compact view of param, 10 best
results[cols].sort_values("rank_test_score").head(10)

,params,mean_test_score,std_test_score,rank_test_score
2,"{'criterion': 'entropy', 'max_features': 'sqrt'}",0.502756,0.004189,1
4,"{'criterion': 'log_loss', 'max_features': 'sqrt'}",0.502637,0.004699,2
5,"{'criterion': 'log_loss', 'max_features': 'log2'}",0.500622,0.005562,3
0,"{'criterion': 'gini', 'max_features': 'sqrt'}",0.500385,0.002664,4
3,"{'criterion': 'entropy', 'max_features': 'log2'}",0.500296,0.003306,5
1,"{'criterion': 'gini', 'max_features': 'log2'}",0.495674,0.003855,6


In [121]:
#SVM_(SVC) Grid

from sklearn.svm import SVC

p_grid = {
    "C":[1.0], 
    "kernel":['rbf'], 
    "degree":[3], 
    "gamma":['scale'], 
    "coef0":[0.0], 
    "shrinking":[True], 
    "probability":['deprecated'], 
    "max_iter":[-1], 
    "decision_function_shape":['ovr'], 
    "break_ties":[False], 
}

SVM_grid = HalvingGridSearchCV(
    estimator=SVC(),
    param_grid=p_grid,
    scoring='accuracy',
    max_resources=500,
    n_jobs=2)

data_training_X = data_training.loc[ : , data_training.columns != "Accident_Severity"]
data_training_Y = data_training["Accident_Severity"]

if(SVM):
    SVM_grid.fit(data_training_X,data_training_Y)
    results = pandas.DataFrame(SVM_grid.cv_results_)

# Compact view of param, 10 worst
results[cols].sort_values("rank_test_score").tail(10)

,params,mean_test_score,std_test_score,rank_test_score
2,"{'criterion': 'entropy', 'max_features': 'sqrt'}",0.502756,0.004189,1
4,"{'criterion': 'log_loss', 'max_features': 'sqrt'}",0.502637,0.004699,2
5,"{'criterion': 'log_loss', 'max_features': 'log2'}",0.500622,0.005562,3
0,"{'criterion': 'gini', 'max_features': 'sqrt'}",0.500385,0.002664,4
3,"{'criterion': 'entropy', 'max_features': 'log2'}",0.500296,0.003306,5
1,"{'criterion': 'gini', 'max_features': 'log2'}",0.495674,0.003855,6


In [122]:
# Compact view of param, 10 best
results[cols].sort_values("rank_test_score").head(10)

,params,mean_test_score,std_test_score,rank_test_score
2,"{'criterion': 'entropy', 'max_features': 'sqrt'}",0.502756,0.004189,1
4,"{'criterion': 'log_loss', 'max_features': 'sqrt'}",0.502637,0.004699,2
5,"{'criterion': 'log_loss', 'max_features': 'log2'}",0.500622,0.005562,3
0,"{'criterion': 'gini', 'max_features': 'sqrt'}",0.500385,0.002664,4
3,"{'criterion': 'entropy', 'max_features': 'log2'}",0.500296,0.003306,5
1,"{'criterion': 'gini', 'max_features': 'log2'}",0.495674,0.003855,6
